# Retail Holiday Effect Analysis

## Project Overview
Analyze retail sales around holidays and promotional periods to measure holiday lift and demand shifts.

## Learning Objectives
- Quantify sales lift during holiday and promotional windows
- Detect pre- and post-holiday demand shifts (pull-forward and hangover effects)
- Compare holiday impact across product categories
- Analyze year-over-year holiday performance

## Problem Statement
Retail planners need to understand exactly how much holidays and promotions lift sales — and how demand shifts around these events — to optimize inventory, staffing, and marketing spend.

## Why This Project Matters
Holiday periods can generate 30-50% of annual retail profit. Precisely measuring holiday effects prevents both stockouts during peaks and excess inventory afterward.

## Dataset Overview
Synthetic daily retail sales: 3 years × 3 product categories with holiday flags, promotional events, and baseline demand.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_days = 365 * 3
dates = pd.date_range('2021-01-01', periods=n_days, freq='D')
categories = ['Electronics', 'Apparel', 'Grocery']

# Define holiday windows
holidays = {
    'ValentinesDay': {'month': 2, 'day': 14, 'window': 5, 'lift': {'Electronics': 0.3, 'Apparel': 0.6, 'Grocery': 0.15}},
    'Easter': {'month': 4, 'day': 10, 'window': 4, 'lift': {'Electronics': 0.1, 'Apparel': 0.3, 'Grocery': 0.4}},
    'July4th': {'month': 7, 'day': 4, 'window': 3, 'lift': {'Electronics': 0.2, 'Apparel': 0.15, 'Grocery': 0.5}},
    'BackToSchool': {'month': 8, 'day': 20, 'window': 14, 'lift': {'Electronics': 0.5, 'Apparel': 0.7, 'Grocery': 0.1}},
    'BlackFriday': {'month': 11, 'day': 26, 'window': 5, 'lift': {'Electronics': 1.5, 'Apparel': 1.0, 'Grocery': 0.3}},
    'Christmas': {'month': 12, 'day': 25, 'window': 14, 'lift': {'Electronics': 1.2, 'Apparel': 0.8, 'Grocery': 0.6}},
}

rows = []
for cat in categories:
    base = {'Electronics': 15000, 'Apparel': 10000, 'Grocery': 25000}[cat]
    t = np.arange(n_days)
    # Weekly seasonality
    weekly = 0.15 * np.sin(2 * np.pi * t / 7)
    # Annual seasonality
    annual = 0.10 * np.sin(2 * np.pi * (t - 80) / 365.25)
    trend = 1 + t * 0.0001

    baseline = base * trend * (1 + weekly + annual)

    # Apply holiday lifts
    holiday_multiplier = np.ones(n_days)
    holiday_flags = ['Normal'] * n_days
    for hname, hinfo in holidays.items():
        for yr in [2021, 2022, 2023]:
            try:
                center = (pd.Timestamp(yr, hinfo['month'], hinfo['day']) - dates[0]).days
                w = hinfo['window']
                for offset in range(-w, w + 1):
                    idx = center + offset
                    if 0 <= idx < n_days:
                        dist_factor = 1 - abs(offset) / (w + 1)
                        lift = hinfo['lift'][cat] * dist_factor
                        holiday_multiplier[idx] = max(holiday_multiplier[idx], 1 + lift)
                        if dist_factor > 0.3:
                            holiday_flags[idx] = hname
            except Exception:
                pass

    # Post-holiday hangover (5 days after major holidays)
    for hname in ['BlackFriday', 'Christmas']:
        for yr in [2021, 2022, 2023]:
            try:
                center = (pd.Timestamp(yr, holidays[hname]['month'], holidays[hname]['day']) - dates[0]).days
                for offset in range(holidays[hname]['window'] + 1, holidays[hname]['window'] + 6):
                    idx = center + offset
                    if 0 <= idx < n_days:
                        holiday_multiplier[idx] *= 0.8
                        holiday_flags[idx] = f'Post-{hname}'
            except Exception:
                pass

    sales = np.clip(baseline * holiday_multiplier + np.random.normal(0, base * 0.05, n_days), 500, None).round(0)
    for i in range(n_days):
        rows.append({'Date': dates[i], 'Category': cat, 'Sales': int(sales[i]),
                     'HolidayFlag': holiday_flags[i]})

df = pd.DataFrame(rows)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.day_name()
print(f'Dataset: {df.shape}')
print(f'Holiday periods: {(df["HolidayFlag"] != "Normal").sum()} flagged days')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nHoliday flag distribution:')
print(df['HolidayFlag'].value_counts())

## Sales Trend with Holiday Markers

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
elec = df[df['Category'] == 'Electronics'].set_index('Date')
ax.plot(elec.index, elec['Sales'], alpha=0.5, linewidth=0.5, label='Electronics Daily')
ax.plot(elec.index, elec['Sales'].rolling(14).mean(), color='red', linewidth=1.5, label='14-day MA')
# Mark major holidays
for hname, color in [('BlackFriday', 'black'), ('Christmas', 'green')]:
    h_dates = elec[elec['HolidayFlag'] == hname].index
    if len(h_dates) > 0:
        ax.scatter(h_dates, elec.loc[h_dates, 'Sales'], color=color, s=20, zorder=5, label=hname)
ax.set_title('Electronics Sales with Holiday Markers')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'sales_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Holiday Lift by Category

In [ ]:
# Calculate lift: holiday avg / non-holiday avg
normal_avg = df[df['HolidayFlag'] == 'Normal'].groupby('Category')['Sales'].mean()
holiday_names = [h for h in df['HolidayFlag'].unique() if h != 'Normal' and not h.startswith('Post-')]
lift_data = {}
for hname in holiday_names:
    h_avg = df[df['HolidayFlag'] == hname].groupby('Category')['Sales'].mean()
    lift_data[hname] = ((h_avg / normal_avg - 1) * 100).round(1)
lift_df = pd.DataFrame(lift_data).T
print('Holiday Lift (% above normal):')
print(lift_df)

fig, ax = plt.subplots(figsize=(12, 6))
lift_df.plot.bar(ax=ax, edgecolor='black')
ax.set_title('Holiday Sales Lift by Category (%)')
ax.set_ylabel('Lift vs Normal (%)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'holiday_lift.png'), dpi=100, bbox_inches='tight')
plt.show()

## Post-Holiday Hangover Effect

In [ ]:
post_flags = [f for f in df['HolidayFlag'].unique() if f.startswith('Post-')]
if post_flags:
    post_avg = df[df['HolidayFlag'].isin(post_flags)].groupby('Category')['Sales'].mean()
    hangover = ((post_avg / normal_avg - 1) * 100).round(1)
    print('Post-Holiday Hangover (% vs Normal):')
    print(hangover)
    fig, ax = plt.subplots(figsize=(8, 5))
    hangover.plot.bar(ax=ax, edgecolor='black', color='salmon')
    ax.set_title('Post-Holiday Sales Drop (%)')
    ax.set_ylabel('Change vs Normal (%)')
    ax.tick_params(axis='x', rotation=0)
    ax.axhline(0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'hangover_effect.png'), dpi=100, bbox_inches='tight')
    plt.show()

## YoY Holiday Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, hname in zip(axes, ['BlackFriday', 'Christmas']):
    for yr in [2021, 2022, 2023]:
        sub = df[(df['Category'] == 'Electronics') & (df['Year'] == yr) & (df['HolidayFlag'] == hname)]
        if len(sub) > 0:
            ax.bar(str(yr), sub['Sales'].sum(), edgecolor='black')
    ax.set_title(f'{hname} Electronics Sales by Year')
    ax.set_ylabel('Total Sales ($)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'yoy_holiday.png'), dpi=100, bbox_inches='tight')
plt.show()

## Monthly Sales Heatmap

In [ ]:
pivot = df.groupby(['Category', 'Month'])['Sales'].mean().unstack(level=0)
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Average Daily Sales by Month × Category')
ax.set_ylabel('Month')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'monthly_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Black Friday** delivers the largest lift across all categories — Electronics sees 100%+ lift
- **Christmas** has a longer window but slightly lower peak intensity
- **Back-to-School** disproportionately lifts Electronics and Apparel — targeted campaigns should focus there
- **Post-holiday hangover** is real — sales drop 10-20% below normal after Black Friday and Christmas
- **Grocery** is the most holiday-resistant category — consistent demand with smaller seasonal swings
- **Year-over-year** holiday performance is growing with the trend — absolute dollar lift increases each year

## Limitations
- Synthetic data with simplified holiday model
- No promotional pricing or discount data
- No inventory or stockout effects
- No customer-level purchase data
- Holiday dates are fixed (Easter moves in reality)

## How to Improve This Project
- Add promotional discount data to separate price vs volume effects
- Include inventory and stockout tracking
- Build pre-holiday demand forecasting models
- Add customer segment analysis (new vs repeat)
- Include online vs in-store channel split

## Production Considerations
- Automated pre-holiday inventory planning
- Dynamic staffing schedules tied to holiday calendars
- Post-holiday markdown optimization
- Holiday campaign ROI tracking

## Common Mistakes
- Measuring holiday lift without accounting for demand pull-forward
- Ignoring the post-holiday hangover when calculating net holiday impact
- Comparing holiday sales across years without adjusting for trend growth
- Treating all categories equally in holiday planning

## Mini Challenge / Exercises
1. What is the net holiday impact (lift minus hangover) for Electronics during Christmas?
2. Which holiday has the most balanced lift across all three categories?
3. Calculate the share of total annual sales that occurs during holiday windows.
4. If you could extend Black Friday by 2 days, estimate the additional revenue based on the decay pattern.

## Final Summary / Key Takeaways
- Holiday effects are large, category-specific, and measurable
- Post-holiday hangover partially offsets the peak lift — net impact is what matters
- Different holidays matter for different categories — one-size-fits-all planning fails
- Year-over-year holiday comparisons must account for underlying trend growth
- Precise holiday effect measurement enables better inventory, staffing, and marketing decisions